In [ ]:
import pandas as pd

eventos = pd.read_csv("../data/eventos_completo.csv")
playbooks = pd.read_csv("../data/playbooks.csv")

print(eventos.head())
print(playbooks)

In [ ]:
LIMIAR_SURPRESA = 1.0
LIMIAR_IAN = 0.5

eventos["opera"] = (eventos["surpresa_zscore"].abs() > LIMIAR_SURPRESA) & (eventos["IAN"] > LIMIAR_IAN)

print(eventos[["data", "surpresa_zscore", "IAN", "opera"]])
print(f"\nTotal de eventos: {len(eventos)}")
print(f"Eventos operados: {eventos['opera'].sum()}")

In [ ]:
def determinar_direcao(row, playbooks):
    if not row["opera"]:
        return None
    regra = playbooks[playbooks["indicador"] == row["indicador"]].iloc[0]
    if row["surpresa_zscore"] > 0:
        return regra["direcao_se_surpresa_positiva"]
    else:
        return regra["direcao_se_surpresa_negativa"]

eventos["direcao"] = eventos.apply(lambda row: determinar_direcao(row, playbooks), axis=1)

print(eventos[["data", "surpresa_zscore", "IAN", "opera", "direcao"]])

In [ ]:
eventos["tamanho_posicao"] = eventos["IAN"] * (1 + eventos["ICE"])
eventos.loc[~eventos["opera"], "tamanho_posicao"] = 0

print(eventos[["data", "IAN", "ICE", "opera", "direcao", "tamanho_posicao"]])

In [ ]:
operacoes = eventos[eventos["opera"]]
print(operacoes[["data", "indicador", "surpresa_zscore", "IAN", "ICE", "direcao", "tamanho_posicao"]])
print(f"\nTotal de operações: {len(operacoes)}")